# Replay buffer

> In-memory episode storage that is both a `Writer` (fill it with `World.collect`) and a `Dataset` (train on it), as in stable-worldmodel.

In [ ]:
#| default_exp data.buffer

In [ ]:
#| hide
from nbdev.showdoc import *

Storage is one ring array per column, `max_steps` rows long, allocated from the first episode's
columns. When a new episode would not fit, whole episodes are evicted oldest first; clips never
cross episodes. `ReplayBuffer(...)` goes where a writer goes (`world.collect(writer=buffer, ...)`),
where a dataset goes (`DataLoader(buffer)`: clips of `history_len` steps), samples batches itself
(`buffer.sample(batch_size)`, through a replaceable sampler), and saves its contents in any format
(`buffer.dump(path, format='hdf5')`).

Columns are stored as collected, per-agent axis included: `pixels` `(num_agents, H, W, C)`,
`action` `(num_agents, ...)`, `reward` / `terminated` / `truncated` `(num_agents,)`. Clips are numpy
arrays (no channel-first conversion), and `action` is reshaped to `(history_len, -1)` as in
`Dataset`: with `frameskip=1` that is `(history_len, num_agents)`. Two additions to stable-worldmodel's
buffer: `clip_indices` (so a `GoalDataset` can wrap the buffer) and `metadata` (passed to the writer
by `dump`; `World.collect` fills it with the env name and kwargs, as for files).

In [ ]:
#| export
# Adapted from stable-worldmodel's data package (https://github.com/galilai-group/stable-worldmodel):
#
# MIT License
#
# Copyright (c) 2026 GalilAI-group
#
# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:
#
# The above copyright notice and this permission notice shall be included in all
# copies or substantial portions of the Software.
#
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
# SOFTWARE.

from __future__ import annotations

import logging
from collections import deque
from pathlib import Path
from typing import Any, Callable, Iterable

import numpy as np

from stable_marl.data.dataset import Dataset
from stable_marl.data.format import EPISODE_DATA_KEY, get_format, split_episode_data

Sampler = Callable[[int, 'ReplayBuffer', int, int], Any]

In [ ]:
#| export
def _uniform_sampler(step: int, buffer: ReplayBuffer, batch_size: int, history_len: int) -> np.ndarray:
    n = buffer.num_valid_ends(history_len)
    if n == 0:
        raise RuntimeError(f"ReplayBuffer.sample: no clips of history_len={history_len} available "
                           f"(num_episodes={buffer.num_episodes}, num_steps_stored={buffer.num_steps_stored})")
    return np.random.randint(0, n, size=batch_size)


class ReplayBuffer(Dataset):
    "In-memory ring-storage replay buffer."
    def __init__(
        self,
        max_steps: int, # Capacity in steps; whole episodes are evicted oldest first when a new one would not fit
        history_len: int = 1, # Steps per clip of `__getitem__` (the Dataset's `num_steps`) and default of `sample`
        frameskip: int = 1, # Stride between the steps of a clip; `action` keeps every step, reshaped to `(history_len, frameskip * action_dim)`
        sampler: Sampler | None = None, # `fn(step, buffer, batch_size, history_len) -> indices` in `[0, buffer.num_valid_ends(history_len))` (default: uniform)
        transform: Callable[[dict], dict] | None = None, # Dict-in / dict-out transform applied to each clip of `__getitem__`
        key_filter: Callable[[dict], dict] | None = None # `fn(ep_data) -> ep_data` choosing the columns stored (e.g. `classic_filter`)
    ):
        for name, value in (('max_steps', max_steps), ('history_len', history_len), ('frameskip', frameskip)):
            if value <= 0:
                raise ValueError(f"{name} must be positive, got {value}")
        self.max_steps, self.history_len, self.frameskip = int(max_steps), int(history_len), int(frameskip)
        self.num_steps = self.history_len
        self.span = self.num_steps * self.frameskip
        self.transform, self.key_filter = transform, key_filter
        self.sampler: Sampler = sampler if sampler is not None else _uniform_sampler
        self.metadata: dict[str, Any] = {}
        self._cols: dict[str, np.ndarray] = {}
        self._head = self._size = 0
        self._episodes: deque[tuple[int, int]] = deque()       # (ring start, length)
        self._episode_meta: deque[dict] = deque()              # episode-scoped data, aligned with _episodes
        self._sample_step = 0
        self._clip_starts: np.ndarray | None = None
        self._clip_starts_span: int | None = None

    # ---- Writer
    def __enter__(self) -> ReplayBuffer:
        return self

    def __exit__(self, *exc) -> None:
        return None

    def write_episode(self, ep_data: dict) -> None:
        "Append one finished episode (every column spanning the whole episode)."
        if not ep_data:
            return
        ep_data, episode_data = split_episode_data(ep_data)
        if self.key_filter is not None:
            ep_data = self.key_filter(ep_data)
        if not ep_data:
            return
        per_step, ep_len = self._coerce_episode(ep_data)
        if ep_len == 0:
            return
        if ep_len > self.max_steps:
            raise ValueError(f"ReplayBuffer.write_episode: episode length {ep_len} exceeds max_steps={self.max_steps}")
        self._ensure_allocated(per_step)
        self._evict_to_fit(ep_len)
        self._append(per_step, ep_len, episode_data)

    def write_episodes(self, episodes: Iterable[dict]) -> None:
        for ep in episodes:
            self.write_episode(ep)

    # ---- contents
    @property
    def column_names(self) -> list[str]:
        return list(self._cols)

    @property
    def episode_column_names(self) -> list[str]:
        seen, cols = set(), []
        for extra in self._episode_meta:
            for k in extra:
                if k not in seen:
                    seen.add(k); cols.append(k)
        return cols

    def get_episode_data(self, episodes_idx: np.ndarray | list[int] | None = None) -> dict[str, list]:
        cols = self.episode_column_names
        if not cols:
            return {}
        metas = list(self._episode_meta)
        idxs = range(len(metas)) if episodes_idx is None else [int(i) for i in np.asarray(episodes_idx, dtype=np.int64).reshape(-1)]
        return {k: [metas[i].get(k) for i in idxs] for k in cols}

    @property
    def num_episodes(self) -> int:
        return len(self._episodes)

    @property
    def num_steps_stored(self) -> int:
        return self._size

    @property
    def lengths(self) -> np.ndarray:
        return np.asarray([ln for _, ln in self._episodes], dtype=np.int32)

    @property
    def offsets(self) -> np.ndarray:
        lens = self.lengths
        if len(lens) == 0:
            return np.zeros(0, dtype=np.int64)
        return np.concatenate(([0], np.cumsum(lens[:-1]))).astype(np.int64)

    @property
    def clip_indices(self) -> list[tuple[int, int]]:
        "`(episode, start)` of every clip of `__getitem__`, in order (as `Dataset`)."
        return [(ep, start) for ep, (_, ln) in enumerate(self._episodes) for start in range(max(0, ln - self.span + 1))]

    def num_valid_ends(self, history_len: int | None = None) -> int:
        "Number of clips of `history_len` steps (the index space the sampler draws from)."
        h = self.history_len if history_len is None else history_len
        return int(self._get_clip_starts(h * self.frameskip)[-1])

    # ---- sampling
    def sample(self, batch_size: int, history_len: int | None = None, step: int | None = None) -> dict[str, np.ndarray]:
        """
        A batch of clips drawn by the sampler: `{column: (batch_size, history_len, ...)}` numpy
        arrays (`transform` is not applied). `step` is passed to the sampler (default: a counter
        of the calls).
        """
        if batch_size <= 0:
            raise ValueError(f"batch_size must be positive, got {batch_size}")
        h = self.history_len if history_len is None else history_len
        if h <= 0:
            raise ValueError(f"history_len must be positive, got {h}")
        flat = self._call_sampler(self._consume_step(step), batch_size, h)
        ep_idx, local_start = self._flat_to_clip(flat, h * self.frameskip)
        episodes = list(self._episodes)
        clips = [self._gather_clip(episodes[int(e)][0], int(s), h) for e, s in zip(ep_idx, local_start)]
        return {col: np.stack([c[col] for c in clips], axis=0) for col in self._cols}

    # ---- Dataset
    def __len__(self) -> int:
        return self.num_valid_ends(self.history_len)

    def __repr__(self) -> str:
        return (f"ReplayBuffer(episodes={self.num_episodes}, steps={self.num_steps_stored}/{self.max_steps}, "
                f"history_len={self.history_len}, frameskip={self.frameskip}, columns=[{', '.join(self.column_names) or '-'}])")

    def __getitem__(self, idx: int) -> dict:
        n = len(self)
        if n == 0:
            raise IndexError("ReplayBuffer is empty")
        if idx < 0:
            idx += n
        if not 0 <= idx < n:
            raise IndexError(idx)
        ep_idx, local_start = self._flat_to_clip(np.array([idx], dtype=np.int64), self.span)
        ring_start, _ = self._episodes[int(ep_idx[0])]
        clip = self._gather_clip(ring_start, int(local_start[0]), self.history_len)
        return self.transform(clip) if self.transform is not None else clip

    def _load_slice(self, ep_idx: int, start: int, end: int) -> dict:
        if not 0 <= ep_idx < self.num_episodes:
            raise IndexError(ep_idx)
        ring_start, ep_len = self._episodes[ep_idx]
        if not 0 <= start <= end <= ep_len:
            raise IndexError(f"slice [{start}:{end}] out of episode of length {ep_len}")
        positions = (ring_start + np.arange(start, end)) % self.max_steps
        return {col: arr[positions] for col, arr in self._cols.items()}

    def get_col_data(self, col: str) -> np.ndarray:
        "The whole column, episodes oldest first."
        return np.concatenate([self._cols[col][(start + np.arange(ln)) % self.max_steps] for start, ln in self._episodes]) \
            if self._episodes else self._cols[col][:0]

    # ---- export
    def episodes(self) -> Iterable[dict]:
        """
        The stored episodes as `{column: [row, ...]}` (what `World.collect` writes and every
        `Writer` takes), with their episode-scoped data under `EPISODE_DATA_KEY`.
        """
        for (ring_start, ep_len), extra in zip(list(self._episodes), list(self._episode_meta)):
            positions = (ring_start + np.arange(ep_len)) % self.max_steps
            ep = {col: list(arr[positions]) for col, arr in self._cols.items()}
            if extra:
                ep[EPISODE_DATA_KEY] = dict(extra)
            yield ep

    def dump(self, path: str | Path, format: str, mode: str = 'overwrite', **kwargs: Any) -> None:
        """
        Save the contents with the writer of `format` (`kwargs` go to it), with `metadata`.
        Episode-scoped data is dropped, with a warning, if the format does not store it.
        """
        fmt = get_format(format)
        episodes: Iterable[dict] = self.episodes()
        if any(self._episode_meta) and not getattr(fmt, 'supports_episode_data', False):
            logging.warning("ReplayBuffer.dump: format '%s' does not support episode data; dropping it (columns %s).",
                            format, self.episode_column_names)
            episodes = ({k: v for k, v in ep.items() if k != EPISODE_DATA_KEY} for ep in episodes)
        with fmt.open_writer(path, mode=mode, **kwargs) as writer:
            if hasattr(writer, 'metadata'):
                writer.metadata.update(self.metadata)
            writer.write_episodes(episodes)

    def clear(self) -> None:
        "Drop every episode (the allocated arrays are reused)."
        self._head = self._size = 0
        self._episodes.clear()
        self._episode_meta.clear()
        self._invalidate_clip_cache()

    # ---- internals
    def _coerce_episode(self, ep_data: dict) -> tuple[dict[str, np.ndarray], int]:
        per_step, ep_len = {}, None
        for col, vals in ep_data.items():
            arr = np.asarray(vals)
            if arr.ndim == 0:
                raise ValueError(f"ReplayBuffer.write_episode: column '{col}' is scalar; expected a per-step array")
            if ep_len is None:
                ep_len = arr.shape[0]
            elif arr.shape[0] != ep_len:
                raise ValueError(f"ReplayBuffer.write_episode: column '{col}' has length {arr.shape[0]}, expected {ep_len}")
            per_step[col] = arr
        return per_step, ep_len or 0

    def _ensure_allocated(self, per_step: dict[str, np.ndarray]) -> None:
        if not self._cols:
            for col, arr in per_step.items():
                self._cols[col] = np.empty((self.max_steps, *arr.shape[1:]), dtype=arr.dtype)
            return
        existing, incoming = set(self._cols), set(per_step)
        if existing != incoming:
            raise ValueError(f"ReplayBuffer.write_episode: schema mismatch. missing={sorted(existing - incoming)}, "
                             f"extra={sorted(incoming - existing)}")
        for col, arr in per_step.items():
            if arr.shape[1:] != self._cols[col].shape[1:]:
                raise ValueError(f"ReplayBuffer.write_episode: column '{col}' per-step shape {arr.shape[1:]} "
                                 f"mismatches expected {self._cols[col].shape[1:]}")

    def _evict_to_fit(self, ep_len: int) -> None:
        evicted = False
        while self._size + ep_len > self.max_steps and self._episodes:
            _, n = self._episodes.popleft()
            self._episode_meta.popleft()
            self._size -= n
            evicted = True
        if evicted:
            self._invalidate_clip_cache()

    def _append(self, per_step: dict[str, np.ndarray], ep_len: int, episode_data: dict | None = None) -> None:
        positions = (self._head + np.arange(ep_len)) % self.max_steps
        for col, arr in per_step.items():
            self._cols[col][positions] = arr
        self._episodes.append((self._head, ep_len))
        self._episode_meta.append(dict(episode_data or {}))
        self._head = (self._head + ep_len) % self.max_steps
        self._size += ep_len
        self._invalidate_clip_cache()

    def _consume_step(self, step: int | None) -> int:
        if step is not None:
            return int(step)
        cur, self._sample_step = self._sample_step, self._sample_step + 1
        return cur

    def _call_sampler(self, step: int, batch_size: int, history_len: int) -> np.ndarray:
        flat = np.asarray(self.sampler(step, self, batch_size, history_len))
        if flat.shape != (batch_size,):
            raise ValueError(f"ReplayBuffer.sample: sampler returned shape {flat.shape}, expected ({batch_size},)")
        return flat.astype(np.int64, copy=False)

    def _flat_to_clip(self, flat: np.ndarray, span: int) -> tuple[np.ndarray, np.ndarray]:
        "Flat clip indices -> `(episode index, clip start in the episode)`."
        starts = self._get_clip_starts(span)
        total = int(starts[-1])
        if flat.size and (flat.min() < 0 or flat.max() >= total):
            raise IndexError(f"ReplayBuffer: clip index out of range [0, {total})")
        ep_idx = np.searchsorted(starts[1:], flat, side='right')
        return ep_idx, flat - starts[ep_idx]

    def _get_clip_starts(self, span: int) -> np.ndarray:
        "Cumulative clip counts per episode, `(N + 1,)`, cached per `span` until the episodes change."
        if self._clip_starts is None or self._clip_starts_span != span:
            n = len(self._episodes)
            valid = np.fromiter((max(0, ln - span + 1) for _, ln in self._episodes), dtype=np.int64, count=n)
            self._clip_starts = np.empty(n + 1, dtype=np.int64)
            self._clip_starts[0] = 0
            np.cumsum(valid, out=self._clip_starts[1:])
            self._clip_starts_span = span
        return self._clip_starts

    def _invalidate_clip_cache(self) -> None:
        self._clip_starts = self._clip_starts_span = None

    def _gather_clip(self, ring_start: int, clip_local_start: int, history_len: int) -> dict[str, np.ndarray]:
        "One clip: columns every `frameskip` steps, `action` dense, reshaped to `(history_len, -1)`."
        base = ring_start + clip_local_start
        obs_idx = (base + np.arange(history_len) * self.frameskip) % self.max_steps
        action_idx = obs_idx if self.frameskip == 1 else (base + np.arange(history_len * self.frameskip)) % self.max_steps
        clip = {col: arr[action_idx if col == 'action' else obs_idx] for col, arr in self._cols.items()}
        if 'action' in clip:
            clip['action'] = clip['action'].reshape(history_len, -1)
        return clip


def classic_filter(ep_data: dict) -> dict:
    """
    `key_filter` keeping the classic replay-buffer columns: `pixels` (each agent's view),
    `action`, `reward`, `terminated` and `truncated`.
    """
    classic_keys = ('pixels', 'action', 'reward', 'terminated', 'truncated')
    return {k: v for k, v in ep_data.items() if k in classic_keys}

In [ ]:
show_doc(ReplayBuffer.write_episode)

In [ ]:
show_doc(ReplayBuffer.num_valid_ends)

In [ ]:
show_doc(ReplayBuffer.sample)

In [ ]:
show_doc(ReplayBuffer.get_col_data)

In [ ]:
show_doc(ReplayBuffer.episodes)

In [ ]:
show_doc(ReplayBuffer.dump)

In [ ]:
show_doc(ReplayBuffer.clear)

### Tests

In [ ]:
import tempfile, torch
from fastcore.test import test_fail
import stable_marl as sm
from stable_marl.data import HDF5Dataset, GoalDataset, EPISODE_DATA_KEY

ENV = dict(agents=3, size=7, num_obstacles=0, n_clutter=0, min_goal_spawn_distance=1, tile_size=4, max_episode_steps=10)

# filled by World.collect: per-agent columns, clips like an HDF5 file of the same episodes
world = sm.World('MultiGrid-FindGoal-15x15-v0', num_envs=2, goal_conditioned=True, **ENV)
world.set_policy(sm.RandomPolicy(seed=0))
buf = ReplayBuffer(max_steps=200, history_len=3)
world.collect(writer=buf, episodes=6, seed=0, progress=False)
assert buf.num_episodes == 6 and buf.metadata['num_agents'] == 3
assert buf._cols['pixels'].shape[1:] == (3, 28, 28, 3) and buf._cols['action'].shape[1:] == (3,)
with tempfile.TemporaryDirectory() as tmp:
    world.set_policy(sm.RandomPolicy(seed=0))         # same actions as for the buffer
    world.collect(f'{tmp}/d.h5', episodes=6, seed=0, progress=False)
    h5 = HDF5Dataset(path=f'{tmp}/d.h5', num_steps=3)
    assert len(buf) == len(h5) and buf.lengths.tolist() == h5.lengths.tolist()
    for i in (0, len(buf) // 2, len(buf) - 1):
        a, b = buf[i], h5[i]
        assert a['action'].shape == (3, 3) and torch.allclose(torch.as_tensor(a['action']), b['action'], equal_nan=True)
        assert np.array_equal(a['pixels'], b['pixels'].numpy()) and np.array_equal(a['position'], b['position'].numpy())
    # dump: the same file, metadata included
    buf.dump(f'{tmp}/dumped.h5', format='hdf5')
    d = HDF5Dataset(path=f'{tmp}/dumped.h5', num_steps=3)
    assert d.metadata['env_name'] == 'MultiGrid-FindGoal-15x15-v0' and len(d) == len(h5)
    assert all(np.array_equal(d.get_col_data(c), h5.get_col_data(c), equal_nan=h5.get_col_data(c).dtype.kind == 'f')
               for c in h5.column_names)
    h5.close(); d.close()

# sampling: (batch, history_len, ...) with the per-agent axis; actions with frameskip
batch = buf.sample(8)
assert batch['pixels'].shape == (8, 3, 3, 28, 28, 3) and batch['reward'].shape == (8, 3, 3)
skip = ReplayBuffer(max_steps=200, history_len=2, frameskip=2)
skip.write_episodes(buf.episodes())
clip = skip[0]
assert clip['action'].shape == (2, 2 * 3) and np.array_equal(clip['step_idx'], [0, 2])

# a GoalDataset over the buffer; classic columns; eviction of whole episodes, oldest first
g = GoalDataset(buf, goal_probabilities=(0.0, 1.0, 0.0, 0.0), seed=0)
item = g[0]
assert item['goal_pixels'].shape == (1, 3, 28, 28, 3) and item['goal_position'][0, :, 0].shape == (3,)
small = ReplayBuffer(max_steps=25, key_filter=classic_filter)
small.write_episodes(buf.episodes())
assert set(small.column_names) == {'pixels', 'action', 'reward', 'terminated', 'truncated'}
assert small.num_steps_stored <= 25 and small.lengths.tolist() == buf.lengths.tolist()[-small.num_episodes:]
test_fail(lambda: small.write_episode({'pixels': np.zeros((30, 3, 1, 1, 3))}), contains='exceeds max_steps')

# episode-scoped data travels with its episode (and is dropped, with a warning, by formats without it)
eb = ReplayBuffer(max_steps=50)
eb.write_episode({'reward': np.zeros((4, 3)), EPISODE_DATA_KEY: {'layout': 'a'}})
eb.write_episode({'reward': np.ones((5, 3)), EPISODE_DATA_KEY: {'layout': 'b'}})
assert eb.get_episode_data() == {'layout': ['a', 'b']} and next(eb.episodes())[EPISODE_DATA_KEY] == {'layout': 'a'}
assert eb.get_col_data('reward').shape == (9, 3)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()